# Replicating Macro Figures from the RSPG Paper

This notebook replicates Figures 2 (panel 3), 3, and 10 from
[*Recurrent Structural Policy Gradient for Partially Observable Mean Field Games*](https://arxiv.org/abs/2602.20141)
for the **Macroeconomics** environment, using SPG and RSPG.

Requires a **GPU runtime** on Colab.

In [ ]:
# --- Setup: clone repo and install dependencies ---
import subprocess, sys, os

# Clone repo if not already present
if not os.path.isdir("/content/mfax/.git"):
    subprocess.check_call(
        ["git", "clone", "https://github.com/ben-moll/mfax.git", "/content/mfax"],
        stderr=subprocess.DEVNULL,
    )
    print("Cloned mfax repo.")
else:
    print("mfax repo already cloned.")

# Check installed JAX version without importing it (avoids stale module cache)
_result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "jax"],
    capture_output=True, text=True,
)
_jax_ver = ""
for _line in _result.stdout.splitlines():
    if _line.startswith("Version:"):
        _jax_ver = _line.split(":", 1)[1].strip()
        break

if not _jax_ver.startswith("0.5"):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "jax[cuda12_pip]==0.5.3", "flax==0.10.5", "optax==0.2.3",
        "gymnax==0.0.8", "tyro==0.7.3", "chex==0.1.86", "distrax==0.1.5",
        "wandb", "matplotlib",
        "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
    ])
    print("Packages installed. Restarting runtime -- please re-run all cells after restart.")
    os.kill(os.getpid(), 9)
else:
    print(f"JAX {_jax_ver} already installed, no restart needed.")

In [ ]:
# --- After restart: set path, verify JAX ---
!nvidia-smi
import sys; sys.path.insert(0, "/content/mfax")
import jax
print(f"JAX version: {jax.__version__}, devices: {jax.devices()}")
assert jax.__version__.startswith("0.5"), (
    f"Expected JAX 0.5.x, got {jax.__version__}. "
    "Please run the cell above and restart when prompted."
)

import os
SAVE_DIR = "/content/figures"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# --- Patch train.py and policy.py for smooth Beta mean evaluation ---
import pathlib

# 1. Add 'mean' method to BetaPolicy in policy.py
policy_path = pathlib.Path("/content/mfax/mfax/utils/nets/policy.py")
policy_code = policy_path.read_text()
if "def mean(self, state, obs):" not in policy_code:
    old = '''    def mode(self, state, obs):
        action_dist = self._action_dist(state, obs)
        action = action_dist.mode()
        action = self.action_loc + action * self.action_scale
        return action'''
    new = old + '''

    def mean(self, state, obs):
        action_dist = self._action_dist(state, obs)
        action = action_dist.mean()
        action = self.action_loc + action * self.action_scale
        return action'''
    policy_path.write_text(policy_code.replace(old, new))
    print("Patched policy.py: added BetaPolicy.mean()")
else:
    print("policy.py already patched")

# 2. Add mf_policy_net and env to TrainResult in train.py
train_path = pathlib.Path("/content/mfax/mfax/train.py")
train_code = train_path.read_text()
if "mf_policy_net: Any = None" not in train_code:
    old_dc = "    final_params: Any"
    new_dc = "    final_params: Any\n    mf_policy_net: Any = None\n    env: Any = None"
    train_code = train_code.replace(old_dc, new_dc)
    old_ret = ("    result.final_eval_results = mf_eval_results\n"
               "    result.final_params = actor_ts.params\n"
               "    return result")
    new_ret = ("    result.final_eval_results = mf_eval_results\n"
               "    result.final_params = actor_ts.params\n"
               "    result.mf_policy_net = mf_policy_net\n"
               "    result.env = env\n"
               "    return result")
    train_code = train_code.replace(old_ret, new_ret)
    train_path.write_text(train_code)
    print("Patched train.py: added mf_policy_net and env to TrainResult")
else:
    print("train.py already patched")

print("Done.")

In [ ]:
# --- Train SPG and RSPG on the Macroeconomics environment ---
from mfax.train import train

overrides = dict(
    task="endogenous", state_type="states", discount_factor=0.95,
    num_envs=8, num_iterations=10000, eval_frequency=50, lr=0.001,
    common_noise=True, normalize_obs=True, normalize_states=True, seed=0,
)

results = {}
for algo in ["spg", "rspg"]:
    results[algo] = train(algo, max_time=1800, **overrides)

## Figure 2, Panel 3: Exploitability vs Training Time (Macroeconomics)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
for algo, style in [("spg", {"color": "C0", "label": "SPG"}),
                     ("rspg", {"color": "C1", "label": "RSPG"})]:
    res = results[algo]
    ax.plot(res.train_times, res.exploitabilities, **style)
ax.set_yscale("log")
ax.set_xlabel("Train Time (s)")
ax.set_ylabel("Exploitability")
ax.set_title("Macroeconomics")
ax.legend()
plt.tight_layout()
fig.savefig(f"{SAVE_DIR}/figure2_panel3_exploitability.png", dpi=150)
plt.show()
print(f"Saved to {SAVE_DIR}/figure2_panel3_exploitability.png")

## Figure 3: Mean-Field Distribution Heatmaps (Macroeconomics)

Rows: SPG, RSPG. Columns: interest rate, wage, MF distribution at t=0, t=64, t=127.

In [ ]:
import numpy as np
import matplotlib.colors as mcolors

fig, axes = plt.subplots(2, 5, figsize=(22, 6))

for row, algo in enumerate(["spg", "rspg"]):
    seq = results[algo].final_eval_results.mf_sequence
    agg = seq.aggregate_s

    # interest rate and wage over time (first env)
    r = np.array(agg.interest_rate[:, 0])
    w = np.array(agg.wage[:, 0])
    axes[row, 0].plot(r)
    axes[row, 1].plot(w)
    if row == 0:
        axes[row, 0].set_title("Interest rate")
        axes[row, 1].set_title("Wage")
    axes[row, 0].set_ylabel(algo.upper())

    # MF distribution heatmaps at selected timesteps
    mu = np.array(agg.mu[:, 0])          # (T, 1000)
    n_wealth, n_income = 200, 5
    mu_2d = mu.reshape(mu.shape[0], n_wealth, n_income)  # (T, wealth, income)
    vmax = mu_2d.max()

    for col, t in enumerate([0, 64, 127]):
        im = axes[row, 2 + col].imshow(
            mu_2d[t].T, aspect="auto", origin="lower",
            norm=mcolors.LogNorm(vmin=max(1e-6, mu_2d[t].max() * 1e-4), vmax=vmax),
        )
        if row == 0:
            axes[row, 2 + col].set_title(f"t = {t}")
        axes[row, 2 + col].set_xlabel("Wealth")
        if col == 0:
            axes[row, 2 + col].set_ylabel("Income")

plt.tight_layout()
fig.savefig(f"{SAVE_DIR}/figure3_mf_heatmaps.png", dpi=150)
plt.show()
print(f"Saved to {SAVE_DIR}/figure3_mf_heatmaps.png")

## Figure 10: Detailed Macro Policy Plots

Rows: value function, expected action, consumption, reward. Columns: SPG, RSPG.
Lines colored by income level.

In [ ]:
import jax.numpy as jnp
from mfax.envs import make_env as _make_env
_env = _make_env("pushforward/endogenous", common_noise=True)

n_wealth, n_income = _env.params.num_states  # (200, 5)
wealth_grid = np.array(_env.params.discrete_states[0])   # (200,)
income_grid = np.array(_env.params.discrete_states[1])    # (5,)
sigma = 2.0  # CRRA coefficient of relative risk aversion

row_labels = ["Value function", "Expected action", "Consumption", "Reward"]
fig, axes = plt.subplots(4, 2, figsize=(12, 16))

for col, algo in enumerate(["spg", "rspg"]):
    res = results[algo]
    ev = res.final_eval_results
    seq = ev.mf_sequence
    env = res.env
    beta_policy = res.mf_policy_net.policy
    params = res.final_params

    # --- Value function (from eval results) ---
    disc_ret = np.array(ev.policy_disc_returns[0]).reshape(n_wealth, n_income)

    # --- Smooth expected action from continuous Beta policy ---
    individual_states = env.params.states  # (1000, 2)
    norm_states = env.normalize_individual_s(individual_states, True)
    agg_obs_t0 = jnp.array(seq.aggregate_obs[0, 0])
    norm_obs = env.normalize_obs(agg_obs_t0[None, :], True)[0]
    broadcasted_obs = res.mf_policy_net._broadcast_aggregate_obs(norm_states, norm_obs)
    mean_action = beta_policy.apply(params, norm_states, broadcasted_obs, method="mean")
    exp_action = np.array(mean_action).squeeze(-1).reshape(n_wealth, n_income)

    # --- Consumption and reward at the mean action ---
    r_t0 = float(np.array(seq.aggregate_s.interest_rate[0, 0]))
    w_t0 = float(np.array(seq.aggregate_s.wage[0, 0]))
    budget = (1 + r_t0) * wealth_grid[:, None] + w_t0 * income_grid[None, :]
    consumption = exp_action * budget
    exp_reward = consumption ** (1 - sigma) / (1 - sigma)

    data = [disc_ret, exp_action, consumption, exp_reward]
    for row_idx, (d, label) in enumerate(zip(data, row_labels)):
        for inc_idx in range(n_income):
            axes[row_idx, col].plot(
                wealth_grid, d[:, inc_idx],
                label=f"y={income_grid[inc_idx]:.2f}" if col == 0 else None,
            )
        if col == 0:
            axes[row_idx, col].set_ylabel(label)
        axes[row_idx, col].set_xlabel("Wealth")
    axes[0, col].set_title(algo.upper())

axes[0, 0].legend(fontsize=7, title="Income")
plt.tight_layout()
fig.savefig(f"{SAVE_DIR}/figure10_macro_policy.png", dpi=150)
plt.show()
print(f"Saved to {SAVE_DIR}/figure10_macro_policy.png")

In [ ]:
# --- Optionally save to Google Drive ---
# Uncomment the lines below to save figures and notebook to Drive:
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil, os
# DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks"
# os.makedirs(DRIVE_DIR, exist_ok=True)
# for f in os.listdir(SAVE_DIR):
#     shutil.copy(f"{SAVE_DIR}/{f}", f"{DRIVE_DIR}/{f}")
# shutil.copy("/content/mfax/notebooks/replicate_figures.ipynb", f"{DRIVE_DIR}/replicate_figures.ipynb")
# print(f"All files saved to {DRIVE_DIR}/")

print(f"Figures saved to {SAVE_DIR}/")
print("To download: click the folder icon in the left sidebar, navigate to /content/figures/")